# Geolocation - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim, create_map, lit

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_geolocation"
target_table = f"{catalog}.silver.olist_geolocation"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: decimal(16,14) (nullable = true)
 |-- geolocation_lng: decimal(17,14) (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation
01046,-23.54608112703554,-46.64482029837157,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation
01041,-23.54439216486810,-46.63949930627844,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation
01035,-23.54157796171149,-46.64160722329613,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation
01012,-23.54776230336427,-46.63536053788448,são paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation
01047,-23.54627311241268,-46.64122516971552,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation
01013,-23.54692320843672,-46.63426369649150,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation
01029,-23.54376905576913,-46.63427784085132,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation
01011,-23.54763955032063,-46.63603162315495,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation


In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

Number of rows: 1000163
Number of columns: 12


In [0]:
for column, dtype in bronze_df.dtypes[:5]:
    print(column)
    print("Null count:", bronze_df.filter(col(column).isNull()).count())
    print("Distinct count:", bronze_df.filter(col(column).isNotNull()).select(column).distinct().count())

    if dtype == "string":
        print("Extra whitespace row count:",
            (
                bronze_df.withColumn(f"{column}_trimmed", trim(col(column)))
                .filter(col(column) != col(f"{column}_trimmed"))
                .count()
            )
        )
    print("-"*20)

geolocation_zip_code_prefix
Null count: 0
Distinct count: 19015
Extra whitespace row count: 0
--------------------
geolocation_lat
Null count: 0
Distinct count: 717334
--------------------
geolocation_lng
Null count: 0
Distinct count: 717614
--------------------
geolocation_city
Null count: 0
Distinct count: 8011
Extra whitespace row count: 1
--------------------
geolocation_state
Null count: 0
Distinct count: 27
Extra whitespace row count: 0
--------------------


In [0]:
duplicate_rows_df = (
    bronze_df
    .groupBy("geolocation_zip_code_prefix",
             "geolocation_lat",
             "geolocation_lng",
             "geolocation_city",
             "geolocation_state"
             )
    .count()
    .filter(col("count") > 1)
)

print(duplicate_rows_df.count())

128174


- No single column is a valid key.
- The combination of all five columns is also not unique.

In [0]:
display(
bronze_df.withColumn("geolocation_city_trimmed", trim(col("geolocation_city")))
.filter(col("geolocation_city") != col("geolocation_city_trimmed"))
)

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset,geolocation_city_trimmed
40243,-12.98781377358929,-38.50574187032669,salvador,BA,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation,salvador


In [0]:
rescued_row_count = (
    bronze_df
    .filter(col("_rescued_data").isNotNull())
    .filter(trim(col("_rescued_data")) != "")
    .count()
)

print("Number of rescued rows:", rescued_row_count)

Number of rescued rows: 0


## Transform to Silver

In [0]:
silver_df = bronze_df.withColumn(
    "geolocation_city",
    trim(col("geolocation_city"))
)

In [0]:
brazil_state_map = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AM": "Amazonas",
    "AP": "Amapá",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MG": "Minas Gerais",
    "MS": "Mato Grosso do Sul",
    "MT": "Mato Grosso",
    "PA": "Pará",
    "PB": "Paraíba",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "PR": "Paraná",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RO": "Rondônia",
    "RR": "Roraima",
    "RS": "Rio Grande do Sul",
    "SC": "Santa Catarina",
    "SE": "Sergipe",
    "SP": "São Paulo",
    "TO": "Tocantins"
}

In [0]:
state_map_expr = create_map(
    *[
        item
        for state_code, state_name in brazil_state_map.items()
        for item in (lit(state_code), lit(state_name))
    ]
)

silver_df = silver_df.withColumn(
    "geolocation_state_name",
    state_map_expr[col("geolocation_state")]
)

In [0]:
display(
    silver_df.select("geolocation_state", "geolocation_state_name").distinct()
)

geolocation_state,geolocation_state_name
SP,São Paulo
RN,Rio Grande do Norte
AC,Acre
RJ,Rio de Janeiro
ES,Espírito Santo
MG,Minas Gerais
BA,Bahia
SE,Sergipe
PE,Pernambuco
AL,Alagoas


In [0]:
silver_df = silver_df.dropDuplicates(["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city","geolocation_state"])

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: decimal(16,14) (nullable = true)
 |-- geolocation_lng: decimal(17,14) (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)
 |-- geolocation_state_name: string (nullable = true)



In [0]:
display(silver_table_df.limit(10))

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset,geolocation_state_name
01035,-23.54157796171149,-46.64160722329613,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation,São Paulo
01014,-23.54643534332621,-46.63383023397196,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation,São Paulo
01009,-23.54634051373423,-46.63623850320263,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation,São Paulo
01013,-23.54768632223438,-46.63413976923070,são paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation,São Paulo
01008,-23.54534149209414,-46.63558668567147,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation,São Paulo
01026,-23.53943950375090,-46.63306386663973,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation,São Paulo
01007,-23.54961074049283,-46.63835065445208,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation,São Paulo
01050,-23.54885719288482,-46.64186497030983,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation,São Paulo
01013,-23.54730787177563,-46.63425127935825,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation,São Paulo
01008,-23.54545046655893,-46.63565673674192,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/geolocation/olist_geolocation_dataset.csv,2026-08-02T21:28:23.000Z,2026-08-04T16:13:02.794Z,4efd99ac-160d-490b-83b1-964f60cdaa8e,olist,geolocation,São Paulo


In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Bronze row count: 1000163
Silver row count: 738332


Exact duplicate rows were removed, reducing the row count from 1,000,163 Bronze rows to 738,332 Silver rows.